# Official NVIDIA StyleGAN2-ADA: preparation

Prepared 10 November 2026. **Not executed in Kaggle; no model trained.**

Use the published implementation, not a project-specific GAN architecture or loop.
Original custom-GAN pilot and diagnostic are historical feasibility work.
Attach the original ISIC image dataset and frozen `isic2019_binary_split.csv`.
Only 972 training images (530 benign / 442 malignant) are exported. No image from validation/test enters GAN fitting.

This candidate uses official class-conditional training at 128x128, with no pretrained checkpoint. Existing architecture does not mean already-trained dermoscopy weights. A future checkpoint needs documented training-data provenance, overlap and shape/conditioning compatibility checks.

Run All prepares data and requests an official configuration dry run. Training is disabled in the final cell. Dry run does not test CUDA kernel compilation or learning quality. Modern Kaggle/PyTorch compatibility remains untested; resolve the runtime before training without silently editing upstream architecture.


In [ ]:
from pathlib import Path
import subprocess, sys, json, platform, importlib.util

UPSTREAM_COMMIT = 'd72cc7d041b42ec8e806021a205ed9349f87c6a4'
work = Path('/kaggle/working/stylegan2ada_phase')
assert not work.exists(), 'Choose a fresh work folder; preserve previous evidence'
work.mkdir()
upstream = work / 'stylegan2-ada-pytorch'
subprocess.run(['git','clone','https://github.com/NVlabs/stylegan2-ada-pytorch.git',str(upstream)], check=True)
subprocess.run(['git','checkout','--detach',UPSTREAM_COMMIT], cwd=upstream, check=True)
actual = subprocess.check_output(['git','rev-parse','HEAD'],cwd=upstream,text=True).strip()
assert actual == UPSTREAM_COMMIT
required = ['torch','click','requests','scipy','psutil','tqdm','PIL','ninja','imageio','imageio_ffmpeg']
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, f'Install official dependencies in a compatible runtime first: {missing}'
import torch
environment = dict(python=platform.python_version(), torch=torch.__version__,
                   cuda=torch.version.cuda, cuda_available=torch.cuda.is_available(),
                   upstream_commit=actual)
(work/'environment.json').write_text(json.dumps(environment,indent=2))
print(environment)


## Frozen training-data adapter

This cell contains data-validation/export code only. It defines no neural network.
Whole-image Pillow bilinear RGB resize to 128x128 is explicit; no GAN input scaling is done here. The official loader handles scaling. The existing classifier remains RGB /255. This GAN resize implementation differs from the old TensorFlow pilot and is recorded as part of the new recipe.


In [ ]:
"""Export the frozen training subset to the official StyleGAN dataset format.

Data adapter only: no GAN architecture, loss, optimizer or training loop.
"""
import argparse
import hashlib
import io
import json
import zipfile
from pathlib import Path

import pandas as pd
from PIL import Image

FINGERPRINT = '4f3037e5dc1726e5a534e3a6f155df3e0e0be52229b057659e097483fd0800cd'
MAPPING = {'NV': 0, 'BKL': 0, 'DF': 0, 'VASC': 0, 'MEL': 1, 'BCC': 1, 'SCC': 1}


def verified_train(frame):
    required = {'image_id', 'label', 'group', 'split', 'original_label', 'target'}
    assert required <= set(frame.columns), 'Manifest columns missing'
    digest = hashlib.sha256(frame[['image_id', 'label', 'group', 'split']]
                           .sort_values('image_id').to_csv(index=False).encode()).hexdigest()
    assert digest == FINGERPRINT, 'Frozen split differs'
    assert len(frame) == 1632 and frame.image_id.is_unique
    assert frame.group.notna().all() and frame.groupby('group').split.nunique().max() == 1
    assert frame.original_label.map(MAPPING).equals(frame.target), 'Target mapping differs'
    assert frame.target.map({0: 'benign', 1: 'malignant'}).equals(frame.label)
    assert frame.split.value_counts().to_dict() == {'train': 972, 'test': 331, 'val': 329}
    train = frame[frame.split == 'train'].sort_values('image_id').reset_index(drop=True)
    assert train.target.value_counts().to_dict() == {0: 530, 1: 442}
    return train


def export_dataset(manifest, input_root, output):
    train = verified_train(pd.read_csv(manifest))
    assert not output.exists(), 'Refusing to overwrite existing dataset'
    wanted = set(train.image_id)
    copies = {}
    for p in sorted(input_root.rglob('*')):
        if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and p.stem.upper() in wanted:
            copies.setdefault(p.stem.upper(), []).append(p)
    assert set(copies) == wanted, 'Training image files missing'
    selected = {}
    for image_id, paths in copies.items():
        if len(paths) > 1:
            assert len({hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}) == 1, image_id
        selected[image_id] = paths[0]
    output.parent.mkdir(parents=True, exist_ok=True)
    partial = output.with_name(output.name + '.partial')
    assert not partial.exists(), 'Incomplete previous export exists; choose a new output path'
    labels, evidence = [], []
    with zipfile.ZipFile(partial, 'x', compression=zipfile.ZIP_STORED) as archive:
        for row in train.itertuples(index=False):
            path = selected[row.image_id]
            with Image.open(path) as im:
                im.load()
                # Explicit recipe: whole-image resize, no crop or augmentation.
                resized = im.convert('RGB').resize((128, 128), Image.Resampling.BILINEAR)
            name = f'images/{row.image_id}.png'
            data = io.BytesIO()
            resized.save(data, format='PNG')
            archive.writestr(name, data.getvalue())
            labels.append([name, int(row.target)])
            evidence.append({'image_id': row.image_id, 'target': int(row.target),
                             'source_sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
        archive.writestr('dataset.json', json.dumps({'labels': labels}))
    with zipfile.ZipFile(partial) as archive:
        assert archive.testzip() is None
        assert len(archive.namelist()) == 973
        assert json.loads(archive.read('dataset.json'))['labels'] == labels
    partial.rename(output)
    report = {'dataset': str(output), 'image_count': 972, 'class_counts': {'0': 530, '1': 442},
              'split_sha256': FINGERPRINT,
              'dataset_sha256': hashlib.sha256(output.read_bytes()).hexdigest(),
              'preprocessing': 'Pillow RGB; whole-image bilinear resize to 128x128; uint8 PNG',
              'note': 'Official GAN loader performs its own numeric scaling; CNN preprocessing unchanged',
              'validation_test_image_content_read': False, 'training_sources': evidence}
    output.with_suffix('.audit.json').write_text(json.dumps(report, indent=2))
    print(f'Exported {len(labels)} training images (530 benign / 442 malignant): {output}')
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--manifest', required=True, type=Path)
    parser.add_argument('--input-root', type=Path, default=Path('/kaggle/input'))
    parser.add_argument('--output', required=True, type=Path)
    args = parser.parse_args()
    export_dataset(args.manifest, args.input_root, args.output)


# CLI entry point omitted in notebook.


In [ ]:
manifests = sorted(Path('/kaggle/input').rglob('isic2019_binary_split.csv'))
assert len(manifests) == 1, 'Attach exactly one frozen binary manifest'
data_zip = work/'isic_train128.zip'
audit = export_dataset(manifests[0], Path('/kaggle/input'), data_zip)
assert audit['image_count'] == 972


## Official configuration dry run

Candidate feasibility budget: 20 kimg (20,000 real-image presentations, not 20,000 unique images or 20,000 steps). One GPU, batch 16, seed 142, conditional model, official auto configuration, ADA geometry pipeline (`bg`), no additional dataset mirroring, no pretrained resume, no expensive FID calculation during this smoke candidate. ADA inside GAN training is distinct from classifier conventional augmentation.

This short budget checks feasibility; it cannot establish convergence or clinical validity. Archive all snapshots/grids rather than selecting visually attractive samples. If feasibility succeeds, specify the full run and downstream synthetic budget before executing it.


In [ ]:
command = [sys.executable, str(upstream/'train.py'),
           f'--outdir={work / "runs"}', f'--data={data_zip}',
           '--gpus=1','--cfg=auto','--cond=1','--aug=ada','--augpipe=bg',
           '--mirror=0','--batch=16','--kimg=20','--snap=1',
           '--metrics=none','--seed=142','--workers=2']
(work/'candidate_command.json').write_text(json.dumps(command,indent=2))
result = subprocess.run(command+['--dry-run'], cwd=upstream,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
(work/'dry_run.log').write_text(result.stdout)
print(result.stdout)
assert result.returncode == 0, 'Official configuration dry run failed; resolve before training'


## Explicit training launch — disabled

Do not enable until the runtime, official source/license, exported labels and dry-run configuration have been reviewed. Preserve the upstream commit and all resolved settings. No custom architecture changes are permitted in this candidate. The first real run also tests CUDA extension compilation and memory use.


In [ ]:
RUN_FEASIBILITY = False
if RUN_FEASIBILITY:
    assert torch.cuda.is_available(), 'A compatible GPU runtime is required'
    assert not (work/'runs').exists(), 'Use a fresh experiment directory'
    subprocess.run(command, cwd=upstream, check=True)
else:
    print('Preparation only. No GAN training launched.')


## Evaluate before classifier augmentation

Review fixed per-class grids across checkpoints, output diversity, class conditioning and near-copy/memorization against training images. Requested labels are not independent clinical validation. Archive all checkpoints, unfiltered samples, settings, hashes and runtime evidence. Only after a satisfactory review should a fixed synthetic-data candidate be evaluated with the unchanged CNN and seeds 42/43/44. Validation remains real-only; the 331 test images remain reserved for the final frozen comparison.

References: [Official implementation](https://github.com/NVlabs/stylegan2-ada-pytorch), [paper](https://arxiv.org/abs/2006.06676), [project transition protocol](../docs/stylegan2ada-transition.md).
